In [1]:
import importlib
import subprocess
import sys

# 1. 定义需要检查的包名 (import时的名称 : pip下载时的名称)
required_packages = {
    "cv2": "opencv-python",
    "mediapipe": "mediapipe",
    "pandas": "pandas",
    "numpy": "numpy",
    "tqdm": "tqdm",
    "scipy": "scipy"
}

# 注意：os 和 concurrent.futures 是 Python 标准库，自带的，不需要安装。

def check_and_install():
    print("正在检查环境依赖...\n")
    for import_name, install_name in required_packages.items():
        try:
            # 尝试导入包
            importlib.import_module(import_name)
            print(f"[已存在] {import_name}")
        except ImportError:
            # 如果不存在，调用 pip 进行安装
            print(f"[缺失] 正在安装 {install_name}...")
            try:
                # 使用当前 Python 解释器环境进行安装
                subprocess.check_call([sys.executable, "-m", "pip", "install", install_name, "-i", "https://pypi.tuna.tsinghua.edu.cn/simple"])
                print(f"--- {install_name} 安装成功！")
            except Exception as e:
                print(f"--- {install_name} 安装失败，错误信息: {e}")

    print("\n所有依赖检查完毕！")

# 执行检查
check_and_install()

正在检查环境依赖...

[已存在] cv2
[缺失] 正在安装 mediapipe...
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 17.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.2/79.2 MB 13.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 33.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.5/216.5 kB 26.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 14.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 28.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.2
    Uninstalling numpy-1.24.2:
      Successfully uninstalled numpy-1.24.2
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scipy 1.10.1 requires numpy<1.27.0,>=1.19.5, but you have numpy 2.0.2 which is incompatible.


--- mediapipe 安装成功！
[已存在] pandas
[已存在] numpy
[已存在] tqdm
[已存在] scipy

所有依赖检查完毕！


In [8]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
from concurrent.futures import ProcessPoolExecutor, as_completed
import zipfile

In [10]:
zip_path = 'DAiSEE.zip'
save_path = 'mnt/'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(save_path)
    print(f"解压完成，文件存放在: {save_path}")

OSError: [Errno 122] Disk quota exceeded

In [9]:
# 1. 读取原始 CSV
df = pd.read_csv('clean_metadata.csv')

# 2. 定义 Windows 路径中需要被“切掉”的公共部分
# 公共部分是 D:\Python\DAiSEE\
old_prefix = r'D:\Python\DAiSEE'

# 3. 定义矩池云中数据存放的真实路径
# 假设把数据解压到了 /mnt/DAiSEE_data/ 目录下
new_prefix = 'mnt/DAiSEE'

# ---------------------------------------------------------
# 执行清洗逻辑
# ---------------------------------------------------------

# 第一步：把所有的反斜杠 \ 替换为 Linux 的斜杠 /
df['video_path'] = df['video_path'].str.replace('\\', '/', regex=False)

# 第二步：把 Windows 的前缀替换掉
# 先把 old_prefix 也转成斜杠模式方便匹配
old_prefix_linux = old_prefix.replace('\\', '/')

# 替换路径
df['video_path'] = df['video_path'].str.replace(old_prefix_linux, new_prefix, case=False)

# 4. 检查一下清洗后的前两行
print("清洗后的路径示例：")
print(df['video_path'].iloc[0])

# 5. 保存为新表格
df.to_csv('daisee_metadata_fixed.csv', index=False)
print("\n路径修复完成！请在后续任务中使用 'daisee_metadata_fixed.csv'")

清洗后的路径示例：
mnt/DAiSEE/DataSet/Train/110001/1100011002/1100011002.avi

路径修复完成！请在后续任务中使用 'daisee_metadata_fixed.csv'


In [11]:
# ============================================================
# 1. 配置参数与路径（需要根据实际情况修改）
# ============================================================
INPUT_CSV = 'daisee_metadata_fixed.csv'   # 带有 binary_label 的表格
OUTPUT_FINAL_CSV = 'B_Team_Balanced_Features.csv' # 最终交付的表格
MAX_WORKERS = 8                            # 矩池云核心数，建议设为核心数的80%
TARGET_LEN = 100                           # 每个视频固定提取100帧

# 【核心算法索引 - 不要轻易修改】
LEFT_EYE_IDX = [33, 160, 158, 133, 153, 144]
RIGHT_EYE_IDX = [263, 387, 385, 362, 380, 373]
FACE_3D_IDX = [33, 263, 1, 61, 291, 199] 
FACE_3D_MODEL = np.array([
    [0.0, 0.0, 0.0], [0.0, -63.6, -12.5], [-43.3, 32.7, -26.0],
    [43.3, 32.7, -26.0], [-28.9, -28.9, -20.0], [28.9, -28.9, -20.0]
], dtype=np.float64)

In [12]:
# ============================================================
# 2. 核心数学特征提取函数
# ============================================================
def get_features(landmarks, img_w, img_h):
    """计算单帧的 EAR, MAR 和 Head Pose 欧拉角"""
    # 定义内部欧式距离计算函数
    def dist(p1, p2): return np.linalg.norm(np.array(p1) - np.array(p2))
    
    # --- [1] 计算 EAR (眼睛睁开度) ---
    # 提取左眼和右眼关键点的比例坐标 (x, y)
    le = [[landmarks[i].x, landmarks[i].y] for i in LEFT_EYE_IDX]
    re = [[landmarks[i].x, landmarks[i].y] for i in RIGHT_EYE_IDX]
    
    # 标准 EAR 公式: (纵向距离1 + 纵向距离2) / (2.0 * 横向距离)
    ear_l = (dist(le[1], le[5]) + dist(le[2], le[4])) / (2.0 * dist(le[0], le[3]))
    ear_r = (dist(re[1], re[5]) + dist(re[2], re[4])) / (2.0 * dist(re[0], re[3]))
    ear = (ear_l + ear_r) / 2.0 # 取双眼平均值，提高鲁棒性
    
    # --- [2] 计算 MAR (嘴巴张开度) ---
    # 修正后的坐标提取：13上唇中, 14下唇中, 78左嘴角, 308右嘴角
    tc = [landmarks[13].x, landmarks[13].y]
    bc = [landmarks[14].x, landmarks[14].y]
    lc = [landmarks[78].x, landmarks[78].y]
    rc = [landmarks[308].x, landmarks[308].y]
    
    # MAR 公式: 纵向距离 / 横向距离
    mar = dist(tc, bc) / dist(lc, rc)
    
    # --- [3] 计算头部姿态 (solvePnP) ---
    # 将 MediaPipe 的比例坐标转换为图像的像素坐标 (x * width, y * height)
    img_pts = np.array([
        [landmarks[i].x * img_w, landmarks[i].y * img_h] for i in FACE_3D_IDX
    ], dtype=np.float64)

    # 构造相机内参矩阵 (假设无畸变，使用图像宽度作为焦距近似值)
    focal_length = img_w
    center = (img_w / 2, img_h / 2)
    camera_matrix = np.array([
        [focal_length, 0, center[0]],
        [0, focal_length, center[1]],
        [0, 0, 1]
    ], dtype=np.float64)
    
    dist_coeffs = np.zeros((4, 1)) # 假设无相机畸变

    # 解算旋转向量 (r_vec)
    _, r_vec, _ = cv2.solvePnP(FACE_3D_MODEL, img_pts, camera_matrix, dist_coeffs)
    
    # 将旋转向量转换为旋转矩阵 (rmat)
    rmat, _ = cv2.Rodrigues(r_vec)
    
    # 将旋转矩阵分解为欧拉角 (angles)
    # angles 包含: [Pitch, Yaw, Roll]
    angles, _, _, _, _, _ = cv2.RQDecomp3x3(rmat)
    
    # 返回所有 5 个核心时序特征
    return [ear, mar, angles[0], angles[1], angles[2]]

In [13]:
# ============================================================
# 3. 多进程工作者函数（每个视频片段由一个独立进程处理）
# ============================================================
def video_worker(video_info):
    """
    接收任务包，执行 MediaPipe 处理。
    注意：MediaPipe 的 FaceMesh 对象必须在函数内部初始化，不能作为参数传递。
    """
    v_path, c_id, label = video_info
    
    # 在矩池云运行建议：检查视频路径是否存在
    if not os.path.exists(v_path):
        # 打印错误信息方便排查路径问题
        print(f"警告: 找不到视频文件 {v_path}")
        return [] 

    # 初始化本地 MediaPipe 实例
    with mp.solutions.face_mesh.FaceMesh(
        static_image_mode=False, 
        max_num_faces=1, 
        refine_landmarks=True,
        min_detection_confidence=0.5
    ) as fm:
        cap = cv2.VideoCapture(v_path)
        if not cap.isOpened():
            print(f"错误: 无法打开视频文件 {v_path}")
            return []
            
        # 获取视频总帧数，计算采样步长
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        # 一共 300 帧，我们要 100 帧，步长就是 300 // 100 = 3
        step = max(1, total_frames // TARGET_LEN) 
        
        # 获取视频基本信息
        img_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        img_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        

        sequence_data = [] # 存储该片段的所有帧特征
        last_valid_val = [0.3, 0.0, 0.0, 0.0, 0.0] # 丢包补偿默认值 [EAR, MAR, P, Y, R]
        
        # 改用基于步长的循环
        processed_count = 0
        current_frame_id = 0
        
        while processed_count < TARGET_LEN:
            # 将读取位置设置到对应的采样点
            # 比如第 0, 3, 6, 9... 帧
            cap.set(cv2.CAP_PROP_POS_FRAMES, current_frame_id)
            ret, frame = cap.read()
            
            if not ret: 
                break 
            
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = fm.process(rgb_frame)
            
            if res.multi_face_landmarks:
                feat = get_features(res.multi_face_landmarks[0].landmark, img_w, img_h)
                last_valid_val = feat
            else:
                feat = last_valid_val 
            
            # 这里保存的 frame_count 建议记录为 processed_count (0-99)
            sequence_data.append([c_id, processed_count] + feat + [label])
            
            processed_count += 1
            current_frame_id += step # 加上步长，跳过中间的帧
            
        cap.release()
        
        # --- 时序长度强行对齐 (Padding/Truncate) ---
        # 对于视频损坏的情况，补齐到 100 帧，确保模型不报错，数据清洗后可以忽略这里
        if len(sequence_data) == 0:
            return [] # 彻底处理失败


        if len(sequence_data) < TARGET_LEN:
            # 使用最后一帧特征进行末尾填充
            last_row = sequence_data[-1]
            # 注意需要修改每一行的 frame_index，使其连续
            for i in range(len(sequence_data), TARGET_LEN):
                new_row = last_row.copy()
                new_row[1] = i # 更新帧索引
                sequence_data.append(new_row)
        elif len(sequence_data) > TARGET_LEN:
            # 如果视频过长，进行截断
            sequence_data = sequence_data[:TARGET_LEN]
            
        return sequence_data

In [14]:
# ============================================================
# 4. 数据增强函数（针对“不专注”特征矩阵进行施加）
# ============================================================
def augment_features(data):
    """
    对 (100, 5) 的特征矩阵进行增强
    特征顺序：0:EAR, 1:MAR, 2:Pitch, 3:Yaw, 4:Roll
    """
    # 必须使用 .copy()，否则会修改原始数据导致无法重复增强
    aug_data = data.copy()
    
    # --- [1] 特征噪声：添加高斯噪声 (模拟传感器抖动) ---
    # EAR 和 MAR 是比例值 (0~1)，噪声要小
    aug_data[:, 0] += np.random.normal(0, 0.015, TARGET_LEN) # EAR
    aug_data[:, 1] += np.random.normal(0, 0.01, TARGET_LEN)  # MAR
    
    # --- [2] 特征缩放：随机乘以 0.95~1.05 系数 (模拟个体差异) ---
    aug_data *= np.random.uniform(0.95, 1.05)
    
    # --- [3] 特征扭曲：时间轴拉伸/压缩 (Time Warping) ---
    # 逻辑说明：通过改变采样点的位置，模拟动作变快或变慢
    x_old = np.linspace(0, 1, TARGET_LEN)
    # 创建一个略微扰动的时间轴
    # 通过将原有的线性轴与一个随机偏移量混合，实现局部的快慢变化
    warp_factor = np.random.uniform(0.8, 1.2) # 整体速度缩放系数
    # 生成扭曲后的时间点（原始帧的线性位置）
    # 新时间轴长度随 warp_factor 变化
    x_new = np.linspace(0, 1, int(TARGET_LEN * warp_factor))
    # 利用 np.interp 进行线性插值（每个特征通道独立）
    warped_data = np.zeros((TARGET_LEN, aug_data.shape[1]))
    for i in range(aug_data.shape[1]):
        warped_data[:, i] = np.interp(x_old, x_new, aug_data[:len(x_new), i])
    aug_data = warped_data
    
    # --- [4] 特征通道扰动：模拟遮挡 (Channel Dropout) ---
    # 在 100 帧下，遮挡 10-20 帧以保证增强强度
    mask_len = np.random.randint(10, 20) 
    start = np.random.randint(0, TARGET_LEN - mask_len)
    
    # 随机选择一个角度通道 (Pitch, Yaw, 或 Roll) 进行扰动
    channel_to_mask = np.random.randint(2, 5) 
    # 将该段置为该通道的均值，模拟数据丢失后的平滑填充
    aug_data[start:start+mask_len, channel_to_mask] = np.mean(aug_data[:, channel_to_mask])
    
    return aug_data

In [15]:
# ============================================================
# 5. 主流程 (最终整合版：包含时序增强、样本均衡与数据集分割逻辑)
# 拆分版本：三个独立函数分别输出 CSV
# ============================================================

In [16]:
# ------------------------------------------------------------
# 主函数 1：仅提取原始特征，不进行任何平衡或增强
# ------------------------------------------------------------

def extract_raw_features(input_csv=INPUT_CSV, output_raw_csv="raw_features.csv"):
    # --- [准备阶段] ---
    # 检查输入文件是否存在
    if not os.path.exists(input_csv):
        print(f"错误：找不到输入文件 {input_csv}")
        return
        
    # 读取 A 同学提供的二分类元数据表格
    df_meta = pd.read_csv(input_csv)
    
    # 构造任务包：(视频路径, 剪辑ID, 标签, 分割集标记)
    # 这里的 r['split'] 对应表格中的 Train/Validation/Test
    tasks = [(r['video_path'], r['clip_id'], r['binary_label'], r['split']) for _, r in df_meta.iterrows()]
    
    # --- [第一阶段：原始特征提取] ---
    print(f"第一阶段：开始多进程提取原始特征 (核心数: {MAX_WORKERS})...")
    raw_results = []
    
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as exe:
        # 使用字典建立 future 与 原始任务信息的映射，防止 as_completed 导致的数据错位
        future_to_task = {exe.submit(video_worker, (t[0], t[1], t[2])): t for t in tasks}
        
        # tqdm 显示实时处理进度
        for future in tqdm(as_completed(future_to_task), total=len(future_to_task), desc="Extracting"):
            task_info = future_to_task[future] # 获取该任务原始的 (path, id, label, split)
            try:
                res_sequence = future.result() # 获取 video_worker 返回的 100 帧特征列表
                if res_sequence:
                    # 将结果结构化，并手动关联该视频对应的 split 标记
                    raw_results.append({
                        'clip_id': task_info[1],
                        'label': task_info[2],
                        'split': task_info[3],
                        # 提取特征列 (EAR, MAR, Pitch, Yaw, Roll)，跳过 ID 和 Index 列
                        'data': np.array([row[2:7] for row in res_sequence]) 
                    })
            except Exception as e:
                print(f"视频 {task_info[1]} 处理出错: {e}")

    if not raw_results:
        print("错误：未提取到任何有效特征，请检查视频路径或算法索引。")
        return

    # --- 将原始特征保存为 CSV（未经任何平衡） ---
    print("\n保存原始特征（未平衡）至:", output_raw_csv)
    df_list = []
    for r in raw_results:
        temp_df = pd.DataFrame(r['data'], columns=['EAR', 'MAR', 'Pitch', 'Yaw', 'Roll'])
        temp_df.insert(0, 'sample_id', r['clip_id'])      # 插入样本ID
        temp_df.insert(1, 'frame_idx', range(TARGET_LEN)) # 插入 0-99 的帧索引
        temp_df['label'] = r['label']
        temp_df['split'] = r['split']
        temp_df['source'] = 'original'                   # 标记数据来源为原始
        df_list.append(temp_df)
    
    full_df = pd.concat(df_list, ignore_index=True)
    if full_df.isnull().values.any():
        full_df.fillna(method='ffill', inplace=True)
    full_df.to_csv(output_raw_csv, index=False)
    print(f"原始特征已保存，路径: {os.path.abspath(output_raw_csv)}")
    return raw_results  # 可选返回 raw_results 供后续使用

In [17]:
# ------------------------------------------------------------
# 主函数 2：读取原始特征 CSV，进行数据平衡与增强，输出增强后的 CSV
# ------------------------------------------------------------
def balance_and_augment(input_raw_csv="raw_features.csv", output_aug_csv="augmented_features.csv", aug_factor=15):
    # 读取原始特征 CSV
    df_raw = pd.read_csv(input_raw_csv)
    
    # 按 sample_id 重构每个片段的特征矩阵（假设 TARGET_LEN 已知）
    grouped = df_raw.groupby('sample_id')
    raw_results_reconstructed = []
    for sid, group in grouped:
        group = group.sort_values('frame_idx')
        data = group[['EAR', 'MAR', 'Pitch', 'Yaw', 'Roll']].values
        if len(data) != TARGET_LEN:
            # 补齐或截断（这里补齐用最后一帧填充）
            if len(data) < TARGET_LEN:
                pad_len = TARGET_LEN - len(data)
                last_row = data[-1:]
                data = np.vstack([data, np.repeat(last_row, pad_len, axis=0)])
            else:
                data = data[:TARGET_LEN]
        raw_results_reconstructed.append({
            'clip_id': sid,
            'label': group['label'].iloc[0],
            'split': group['split'].iloc[0],
            'data': data
        })
    
    print(f"重建了 {len(raw_results_reconstructed)} 个片段，开始平衡与增强...")
    
    # --- [第二阶段：数据平衡与针对性增强] ---
    print("\n第二阶段：开始处理数据平衡（仅针对训练集的不专注样本进行增强）...")
    
    # 定义结果容器（使用列表存储 DataFrame，最后统一合并以节省内存）
    final_df_list = []
    
    # 统计计数器
    count_focused = 0
    count_unfocused = 0
    count_augmented = 0

    # 遍历所有重构的原始特征
    for r in tqdm(raw_results_reconstructed, desc="Balancing & Augmenting"):
        # 1. 无论属于哪个 Split，先将原始特征存入
        temp_df = pd.DataFrame(r['data'], columns=['EAR', 'MAR', 'Pitch', 'Yaw', 'Roll'])
        temp_df.insert(0, 'sample_id', r['clip_id'])      # 插入样本ID
        temp_df.insert(1, 'frame_idx', range(TARGET_LEN)) # 插入 0-99 的帧索引
        temp_df['label'] = r['label']
        temp_df['split'] = r['split']
        temp_df['source'] = 'original'                   # 标记数据来源为原始
        final_df_list.append(temp_df)
        
        # 统计原始样本量
        if r['label'] == 0: # 假设 0 代表不专注 (Unfocused)
            count_unfocused += 1
        else:
            count_focused += 1
        
        # 2. 【核心平衡逻辑】：
        # 只有当数据属于 训练集(Train) 且 标签为 不专注(0) 时，才执行 15 倍数据增强
        if str(r['split']).lower() == 'train' and r['label'] == 0:
            for aug_idx in range(aug_factor):
                # 调用第四部分定义的增强函数（包含噪声、缩放、时空扭曲等）
                aug_matrix = augment_features(r['data'])
                
                # 构造增强后的 DataFrame
                aug_df = pd.DataFrame(aug_matrix, columns=['EAR', 'MAR', 'Pitch', 'Yaw', 'Roll'])
                aug_df.insert(0, 'sample_id', f"{r['clip_id']}_aug_{aug_idx}") # 唯一增强ID
                aug_df.insert(1, 'frame_idx', range(TARGET_LEN))
                aug_df['label'] = r['label']
                aug_df['split'] = r['split']
                aug_df['source'] = 'augmented' # 标记数据来源为增强
                final_df_list.append(aug_df)
                count_augmented += 1

    print(f"\n统计结果：")
    print(f" - 原始专注样本 (Label 1): {count_focused} 个")
    print(f" - 原始不专注样本 (Label 0): {count_unfocused} 个")
    print(f" - 训练集增强样本 (Augmented): {count_augmented} 个")

    # --- [第三阶段：合并与 CSV 结构化输出] ---
    print(f"\n第三阶段：正在合并所有特征并保存至 {output_aug_csv}...")
    
    if not final_df_list:
        print("错误：最终数据集为空，请检查逻辑。")
        return

    # 一次性合并所有 DataFrame。这种方式比在循环中使用 append/concat 内存效率高得多
    full_df = pd.concat(final_df_list, ignore_index=True)
    
    # 最后的完整性检查：如果存在 MediaPipe 偶尔失效导致的 NaN，使用前向填充补齐
    if full_df.isnull().values.any():
        full_df.fillna(method='ffill', inplace=True)
    
    # 导出最终 CSV 表格
    full_df.to_csv(output_aug_csv, index=False)
    
    # 输出最终报告
    print(f"\n[任务完成]！")
    print(f"最终总样本数 (含增强): {full_df['sample_id'].nunique()} 个片段")
    print(f"最终数据总行数 (样本数 * 100): {len(full_df)} 行")
    print(f"交付文件路径: {os.path.abspath(output_aug_csv)}")

In [18]:
# ------------------------------------------------------------
# 主函数 3：最终后处理与报告
# ------------------------------------------------------------
def final_merge_and_report(input_aug_csv="augmented_features.csv", output_final_csv="final_features.csv"):
    """
    可选函数：对增强后的 CSV 做最终清洗、生成报告。
    """
    if not os.path.exists(input_aug_csv):
        print(f"错误：找不到增强特征文件 {input_aug_csv}")
        return

    df = pd.read_csv(input_aug_csv)

    # 再次检查 NaN，用前一帧填充
    if df.isnull().values.any():
        df.fillna(method='ffill', inplace=True)

    df.to_csv(output_final_csv, index=False)
    print(f"最终数据集已保存至: {os.path.abspath(output_final_csv)}")

    # 打印统计摘要
    print("\n最终数据集统计：")
    print(f"总样本片段数: {df['sample_id'].nunique()}")
    print(f"总行数: {len(df)}")
    print("各 split 下的标签分布：")
    print(df.groupby(['split', 'label']).size())
    return df

In [ ]:
# ============================================================
# 程序入口
# ============================================================
if __name__ == "__main__":
    extract_raw_features(input_csv=INPUT_CSV, output_raw_csv="raw_features.csv")

第一阶段：开始多进程提取原始特征 (核心数: 8)...


In [ ]:
if __name__ == "__main__":
    balance_and_augment(input_raw_csv="raw_features.csv", output_aug_csv="augmented_features.csv", aug_factor=15)

In [ ]:
if __name__ == "__main__":
    final_merge_and_report(input_aug_csv="augmented_features.csv", output_final_csv="final_features.csv")